# 07 - Silver Layer Quality Check

Observability notebook for the Silver layer. Visualizes quality flag distributions, CPI deflation, geo coverage, and STL decomposition.

**This notebook contains no processing logic** — all transforms happen in `src/medallion/silver.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from config import SILVER_DIR, CPI_INDEX_FILE, STL_COMPONENTS_FILE, MEDALLION_METADATA_DIR, MACRO_SHOCK_YEARS

## Load Silver manifest and data

In [ ]:
import json

manifest_path = MEDALLION_METADATA_DIR / "silver_manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"Silver: {manifest['output_row_count']} rows, geo coverage {manifest['geo_coverage_pct']}%")
    print(f"STL regions: {manifest['stl_regions']}")
else:
    print("No Silver manifest found. Run: make silver")

In [ ]:
silver_path = SILVER_DIR / "transactions_enriched.parquet"
assert silver_path.exists(), f"Silver file not found: {silver_path}"

df = pd.read_parquet(silver_path)
print(f"Shape: {df.shape}")

## Quality flag distributions

In [ ]:
flag_cols = sorted([c for c in df.columns if c.startswith("is_")])
flag_counts = {col: int(df[col].sum()) for col in flag_cols}
flag_pcts = {col: round(count / len(df) * 100, 2) for col, count in flag_counts.items()}

flag_df = pd.DataFrame({"count": flag_counts, "pct": flag_pcts}).sort_values("pct", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
flag_df["pct"].plot(kind="barh", ax=ax, color="coral")
ax.set_xlabel("% of rows flagged")
ax.set_title("Silver: Quality Flag Distribution")
for i, (idx, row) in enumerate(flag_df.iterrows()):
    ax.text(row["pct"] + 0.3, i, f"{row['pct']:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()

flag_df

## CPI chain index

In [ ]:
if CPI_INDEX_FILE.exists():
    cpi = pd.read_parquet(CPI_INDEX_FILE)
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(cpi["quarter_id"], cpi["cpi_index_q"], color="darkgreen", linewidth=1.2)
    ax.set_title("CPI Chain Index (quarterly compounding)")
    ax.set_ylabel("CPI index")
    ax.xaxis.set_major_locator(plt.MaxNLocator(15))
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("CPI index file not found.")

## Geographic coverage

In [ ]:
if "geo_join_status" in df.columns:
    geo_status = df["geo_join_status"].value_counts()
    print("Geo join status:")
    for status, count in geo_status.items():
        print(f"  {status}: {count:,} ({count / len(df) * 100:.1f}%)")
    
    if "lat" in df.columns and "lon" in df.columns:
        matched = df[df["geo_join_status"] == "matched"]
        fig, ax = plt.subplots(figsize=(6, 8))
        ax.scatter(matched["lon"], matched["lat"], s=0.1, alpha=0.3, c="steelblue")
        ax.set_title(f"Geo-matched transactions (n={len(matched):,})")
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_aspect(1.5)
        plt.tight_layout()
        plt.show()
else:
    print("No geo_join_status column — geo join may not have run.")

## STL decomposition

In [ ]:
if STL_COMPONENTS_FILE.exists():
    stl = pd.read_parquet(STL_COMPONENTS_FILE)
    regions = stl["region"].unique()[:4]
    
    fig, axes = plt.subplots(len(regions), 1, figsize=(14, 3 * len(regions)), sharex=True)
    if len(regions) == 1:
        axes = [axes]
    
    for ax, region in zip(axes, regions):
        r = stl[stl["region"] == region].sort_values("quarter_id")
        ax.plot(r["quarter_id"], r["trend"], label="trend", linewidth=1.5)
        ax.fill_between(r["quarter_id"], r["seasonal"], alpha=0.3, label="seasonal")
        ax.set_title(region)
        ax.legend(loc="upper left", fontsize=8)
        ax.xaxis.set_major_locator(plt.MaxNLocator(10))
    
    plt.suptitle("STL Decomposition by Region", y=1.01)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("STL components file not found.")

## Real vs nominal price comparison

In [ ]:
if "real_sqm_price" in df.columns:
    quarterly = df.groupby("quarter_id").agg(
        nominal=pd.NamedAgg(column="sqm_price", aggfunc="median"),
        real=pd.NamedAgg(column="real_sqm_price", aggfunc="median"),
    ).sort_index()
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(quarterly.index, quarterly["nominal"], label="Nominal DKK/m\u00b2", alpha=0.8)
    ax.plot(quarterly.index, quarterly["real"], label="Real DKK/m\u00b2 (2024)", alpha=0.8)
    ax.set_title("Median sqm price: nominal vs CPI-deflated")
    ax.legend()
    ax.xaxis.set_major_locator(plt.MaxNLocator(15))
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No real_sqm_price column — deflation may not have run.")